# Pricing Pro — quick 3-phase training (Colab)

This notebook runs a **small** end-to-end run (SFT data → SFT training → heuristic RL → council RL), writes **`training_metrics.json`**, exports **loss/reward plots** to `docs/`, and copies everything to **`/content/pricing_pro_outputs/`** for download.

**You need (your own keys):**
- **`GROQ_API_KEY`** — for Council / SFT data that calls Groq ([console.groq.com](https://console.groq.com/keys)).
- **GPU runtime** — *Runtime → Change runtime type → T4 (or A100)*.

**Optional:** `HF_TOKEN` if you use gated private models; public Unsloth models usually work without it.

Edit **`GIT_REPO_URL`** in the next cell to your cloneable GitHub (or fork) URL.

In [ ]:
#@title Config
GIT_REPO_URL = "https://github.com/Vedant-1016/Trinity_RL_Final_Economy.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

# Defaults match repo: SFT=5, heuristic=50, council=10 (one metrics row per scenario; loops do not multiply rows)
SFT_SAMPLES = 5  # @param {type:"integer"}
HEURISTIC_SCENARIOS = 50  # @param {type:"integer"}
COUNCIL_SCENARIOS = 10  # @param {type:"integer"}

OUTPUT_DIR = "/content/pricing_pro_outputs"  # saved artifacts (metrics, plots, log)
WORK_DIR = "/content/Trinity_RL_Final_Economy"


In [ ]:
#@title Set API keys (not stored in the notebook — only this runtime)
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")
_hf = getpass("Enter HF_TOKEN (optional, press Enter to skip): ").strip()
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))


In [ ]:
#@title GPU check (must be CUDA for Unsloth training)
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU."
print("Device:", torch.cuda.get_device_name(0))


After the **install** cell finishes, if `import unsloth` fails in training, use **Runtime → Restart session**, then re-run **Config**, **API keys**, **GPU check**, and continue from **Clone** (do not re-run install unless needed).


In [ ]:
#@title Install Unsloth + training stack (Colab; a few minutes)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "trl", "xformers", "bitsandbytes"])
print("Install finished.")


In [ ]:
#@title Clone repository
import os
import shutil
import subprocess
import sys

if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
subprocess.check_call(
    ["git", "clone", "--depth", "1", "-b", BRANCH, GIT_REPO_URL, WORK_DIR]
)
os.chdir(WORK_DIR)
print("CWD:", os.getcwd())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Repo requirements installed.")


In [ ]:
#@title 1) Generate SFT dataset (uses Groq via Council when key is set)
import os
import subprocess
import sys
os.chdir(WORK_DIR)
os.environ["SFT_SAMPLES"] = str(SFT_SAMPLES)
subprocess.check_call([sys.executable, "generate_sft_data.py", "--num-samples", str(SFT_SAMPLES)], cwd=WORK_DIR)
print("Wrote sft_dataset.json")


In [ ]:
#@title 2) Run full training: SFT + heuristic + council (takes a while on T4)
import os
import subprocess
import sys
from datetime import datetime

os.environ["SFT_SAMPLES"] = str(SFT_SAMPLES)
os.environ["HEURISTIC_SCENARIOS"] = str(HEURISTIC_SCENARIOS)
os.environ["COUNCIL_SCENARIOS"] = str(COUNCIL_SCENARIOS)

os.makedirs("docs", exist_ok=True)
log_path = os.path.join(OUTPUT_DIR, f"train_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.log")
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.chdir(WORK_DIR)
print("=== train_llm.py (streaming) ===")
with open(log_path, "w", encoding="utf-8") as log:
    p = subprocess.Popen(
        [sys.executable, "train_llm.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ},
        cwd=WORK_DIR,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    code = p.wait()
if code != 0:
    print("\nWarning: train_llm.py exited with", code, "(metrics/plots may still exist)")


In [ ]:
#@title 3) Export loss / reward plots
import os
import subprocess
import sys

os.chdir(WORK_DIR)
if os.path.exists("training_metrics.json"):
    subprocess.call([sys.executable, "tools/export_training_plots.py"])
    print("Plots saved under docs/")
else:
    print("No training_metrics.json; skip plots.")


In [ ]:
#@title 4) Copy metrics, plots, and logs to a download folder + zip
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
wd = Path(WORK_DIR)
for name in ("training_metrics.json", "sft_dataset.json"):
    p = wd / name
    if p.exists():
        shutil.copy2(p, out / name)
docs = wd / "docs"
if docs.is_dir():
    dest_docs = out / "docs"
    if dest_docs.exists():
        shutil.rmtree(dest_docs)
    shutil.copytree(docs, dest_docs)
zip_path = "/content/pricing_pro_outputs.zip"
shutil.make_archive("/content/pricing_pro_outputs", "zip", OUTPUT_DIR)
print("Folder:", OUTPUT_DIR)
display(FileLink(zip_path))


In [ ]:
#@title 5) Show plots inline (if present)
from IPython.display import Image, display
from pathlib import Path

d = Path(WORK_DIR) / "docs"
for fname in ("loss_curve.png", "reward_curve.png", "baseline_vs_trained.png"):
    p = d / fname
    if p.exists():
        print(fname)
        display(Image(str(p)))
